# AI-related capital expenditure and US GDP

This notebook implements a transparent bottom-up estimate of the **direct measured** contribution of AI-related capital expenditure to US GDP. It starts with quarterly company CAPEX, subtracts a non-AI counterfactual, allocates the resulting investment to the United States and to GDP asset categories, applies recognition lags, estimates associated imports, deflates the separate investment and import flows, and compares published/projected GDP growth with a no-AI counterfactual.

**Headline definitions**

1. **Gross AI-related CAPEX:** positive company-level CAPEX above the 2022 baseline.
2. **US AI investment:** gross AI-related CAPEX after the US share, land exclusion, and recognition lags.
3. **Gross GDP contribution:** GDP growth less growth in a counterfactual that removes AI private fixed investment.
4. **Net measured GDP contribution:** GDP growth less growth in a counterfactual that removes AI investment and reverses its associated import subtraction.

Company CAPEX is quarterly and not seasonally adjusted. The primary results are calendar-year and quarterly year-over-year contributions; q/q SAAR estimates are intentionally not headlined.

**Update workflow:** replace `capex_final_data.csv`, update `LAST_ACTUAL_QUARTER`, optionally set `REFRESH_OFFICIAL_DATA = True`, and run all cells. The notebook creates `data_cache/` and `output/` automatically. A BEA API key is read only from `BEA_API_KEY`; when it is absent, official public BEA bulk workbooks are used.

In [ ]:
# 1. Setup
from pathlib import Path
from datetime import datetime, timezone
import json
import os
import re
import urllib.parse
import urllib.request
import warnings

import numpy as np
import pandas as pd

try:
    import requests
except ImportError:
    requests = None

try:
    import matplotlib.pyplot as plt
    from matplotlib.ticker import PercentFormatter
    HAS_MATPLOTLIB = True
except ImportError:
    plt = None
    PercentFormatter = None
    HAS_MATPLOTLIB = False

try:
    from IPython.display import display, Markdown, Image as IPythonImage
except ImportError:
    def display(value):
        if hasattr(value, "to_string"):
            print(value.to_string())
        else:
            print(value)
    def Markdown(value):
        return value
    IPythonImage = None

ROOT = Path.cwd()
CSV_PATH = ROOT / "capex_final_data.csv"
CACHE_DIR = ROOT / "data_cache"
OUTPUT_DIR = ROOT / "output"
CHART_DIR = OUTPUT_DIR / "charts"
for directory in (CACHE_DIR, OUTPUT_DIR, CHART_DIR):
    directory.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

RUN_UTC = datetime.now(timezone.utc).replace(microsecond=0).isoformat()
print(f"Working directory: {ROOT}")
print(f"Matplotlib available: {HAS_MATPLOTLIB}")

In [ ]:
# 2. ASSUMPTIONS — edit this cell, not the calculation cells
LAST_ACTUAL_QUARTER = pd.Period("2026Q2", freq="Q")
MODEL_START = pd.Period("2022Q1", freq="Q")
ANALYSIS_END = pd.Period("2028Q4", freq="Q")
TAIL_END = pd.Period("2029Q3", freq="Q")

REFRESH_OFFICIAL_DATA = False
EXPORT_OUTPUTS = True
IMPUTATION_WARNING_SHARE = 0.20

US_SHARE_DEFAULT = 0.70
US_SHARE_UTILITY = 1.00
COMPANY_US_SHARE_OVERRIDES = {}

NOMINAL_GDP_GROWTH_ANNUAL = 0.04
GDP_INFLATION_ANNUAL_BOUNDS = (0.01, 0.04)
COMPONENT_PRICE_ANNUAL_BOUNDS = (-0.10, 0.10)
TREND_GROWTH_ANNUAL_BOUNDS = (-0.05, 0.10)

PROFILE_OVERRIDES = {
    "Microsoft Corp": "hyperscaler_cloud",
    "Amazon.com Inc": "hyperscaler_cloud",
    "Alphabet Inc": "hyperscaler_cloud",
    "Meta Platforms Inc": "hyperscaler_cloud",
    "Oracle Corp": "hyperscaler_cloud",
    "CoreWeave Inc": "hyperscaler_cloud",
    "Applied Digital Corp": "hyperscaler_cloud",
    "DigitalOcean Holdings Inc": "hyperscaler_cloud",
    "Cloudflare Inc": "hyperscaler_cloud",
    "Tesla Inc": "tesla_automotive",
}

INDUSTRY_PROFILE = {
    "Software & Services": "software_services",
    "Technology Hardware & Equipment": "tech_hardware",
    "Utilities": "utilities",
    "Media & Entertainment": "hyperscaler_cloud",
    "Automobiles & Components": "tesla_automotive",
    "Consumer Discretionary Distribution & Retail": "hyperscaler_cloud",
}

CATEGORY_SPLITS = pd.DataFrame.from_dict(
    {
        "hyperscaler_cloud": {
            "dc_structures": 0.25, "power_structures": 0.00,
            "it_equipment": 0.55, "power_cooling_equipment": 0.10,
            "other_pfi": 0.05, "land_excluded": 0.05,
        },
        "software_services": {
            "dc_structures": 0.10, "power_structures": 0.00,
            "it_equipment": 0.50, "power_cooling_equipment": 0.15,
            "other_pfi": 0.20, "land_excluded": 0.05,
        },
        "tech_hardware": {
            "dc_structures": 0.05, "power_structures": 0.00,
            "it_equipment": 0.25, "power_cooling_equipment": 0.15,
            "other_pfi": 0.50, "land_excluded": 0.05,
        },
        "utilities": {
            "dc_structures": 0.00, "power_structures": 0.55,
            "it_equipment": 0.00, "power_cooling_equipment": 0.35,
            "other_pfi": 0.05, "land_excluded": 0.05,
        },
        "tesla_automotive": {
            "dc_structures": 0.00, "power_structures": 0.00,
            "it_equipment": 0.05, "power_cooling_equipment": 0.10,
            "other_pfi": 0.80, "land_excluded": 0.05,
        },
    },
    orient="index",
)

LAG_WEIGHTS = {
    "dc_structures": [0.30, 0.30, 0.20, 0.20],
    "power_structures": [0.25, 0.25, 0.25, 0.25],
    "it_equipment": [0.60, 0.30, 0.10],
    "power_cooling_equipment": [0.50, 0.30, 0.20],
    "other_pfi": [0.70, 0.20, 0.10],
}

IMPORT_SHARES = pd.DataFrame(
    {
        "low":  {"dc_structures": 0.00, "power_structures": 0.00, "it_equipment": 0.45, "power_cooling_equipment": 0.15, "other_pfi": 0.15},
        "base": {"dc_structures": 0.00, "power_structures": 0.00, "it_equipment": 0.60, "power_cooling_equipment": 0.30, "other_pfi": 0.25},
        "high": {"dc_structures": 0.00, "power_structures": 0.00, "it_equipment": 0.75, "power_cooling_equipment": 0.45, "other_pfi": 0.35},
    }
)

GS_US_AI_CAPEX_2026_BN = 581.0
GS_AI_CAPEX_SHARE_ANCHORS = {2026: 1.8, 2027: 2.5, 2028: 2.8}

BEA_BULK_FILES = {
    "section1": ("bea_section1.xlsx", "https://apps.bea.gov/national/Release/XLS/Survey/Section1All_xls.xlsx"),
    "section4": ("bea_section4.xlsx", "https://apps.bea.gov/national/Release/XLS/Survey/Section4All_xls.xlsx"),
    "section5": ("bea_section5.xlsx", "https://apps.bea.gov/national/Release/XLS/Survey/Section5All_xls.xlsx"),
    "underlying5": ("bea_underlying_section5.xlsx", "https://apps.bea.gov/national/Release/XLS/Underlying/Section5All_xls.xlsx"),
    "census": ("census_privsatime.xlsx", "https://www.census.gov/construction/c30/xlsx/privsatime.xlsx"),
}

SERIES_REGISTRY = {
    "nominal_gdp": dict(dataset="NIPA", table="T10105", bulk="section1", sheet="T10105-Q", code="A191RC", label="Gross domestic product", scale=0.001),
    "real_gdp": dict(dataset="NIPA", table="T10106", bulk="section1", sheet="T10106-Q", code="A191RX", label="Gross domestic product", scale=0.001),
    "equipment_gdp_contribution": dict(dataset="NIPA", table="T10102", bulk="section1", sheet="T10102-Q", code="Y033RY", label="Equipment", scale=1.0),
    "nominal_pfi": dict(dataset="NIPA", table="T50305", bulk="section5", sheet="T50305-Q", code="A007RC", label="Private fixed investment", scale=0.001),
    "real_pfi": dict(dataset="NIPA", table="T50306", bulk="section5", sheet="T50306-Q", code="A007RX", label="Private fixed investment", scale=0.001),
    "computer_pfi_contribution": dict(dataset="NIPA", table="T50302", bulk="section5", sheet="T50302-Q", code="B935RZ", label="Computers and peripheral equipment", scale=1.0),
    "computers_nominal": dict(dataset="NIPA", table="T50305", bulk="section5", sheet="T50305-Q", code="B935RC", label="Computers and peripheral equipment", scale=0.001),
    "computers_real": dict(dataset="NIPA", table="T50306", bulk="section5", sheet="T50306-Q", code="B935RX", label="Computers and peripheral equipment", scale=0.001),
    "computers_price": dict(dataset="NIPA", table="T50304", bulk="section5", sheet="T50304-Q", code="B935RG", label="Computers and peripheral equipment", scale=1.0),
    "software_price": dict(dataset="NIPA", table="T50304", bulk="section5", sheet="T50304-Q", code="B985RG", label="Software", scale=1.0),
    "other_equipment_price": dict(dataset="NIPA", table="T50304", bulk="section5", sheet="T50304-Q", code="A862RG", label="Other equipment", scale=1.0),
    "nonres_structures_price": dict(dataset="NIPA", table="T50304", bulk="section5", sheet="T50304-Q", code="B009RG", label="Structures", scale=1.0),
    "office_price": dict(dataset="NIUnderlyingDetail", table="U50404", bulk="underlying5", sheet="U50404-Q", code="W031RG", label="Office", scale=1.0),
    "dc_structures_price": dict(dataset="NIUnderlyingDetail", table="U50404", bulk="underlying5", sheet="U50404-Q", code="IA001282", label="Data centers", scale=1.0),
    "dc_structures_nominal": dict(dataset="NIUnderlyingDetail", table="U50405", bulk="underlying5", sheet="U50405-Q", code="LA001282", label="Data centers", scale=0.001),
    "power_structures_price": dict(dataset="NIUnderlyingDetail", table="U50404", bulk="underlying5", sheet="U50404-Q", code="W028RG", label="Power", scale=1.0),
    "communication_equipment_price": dict(dataset="NIUnderlyingDetail", table="U50504", bulk="underlying5", sheet="U50504-Q", code="B275RG", label="Communication equipment", scale=1.0),
    "electrical_equipment_price": dict(dataset="NIUnderlyingDetail", table="U50504", bulk="underlying5", sheet="U50504-Q", code="B276RG", label="Electrical transmission", scale=1.0),
    "computer_imports_nominal": dict(dataset="NIPA", table="T40205B", bulk="section4", sheet="T40205B-Q", code="B852RC", label="Computers, peripherals, and parts", scale=0.001),
    "computer_imports_real": dict(dataset="NIPA", table="T40206B", bulk="section4", sheet="T40206B-Q", code="B852RX", label="Computers, peripherals, and parts", scale=0.001),
    "computer_imports_price": dict(dataset="NIPA", table="T40204B", bulk="section4", sheet="T40204B-Q", code="B852RG", label="Computers, peripherals, and parts", scale=1.0),
}

assert np.allclose(CATEGORY_SPLITS.sum(axis=1), 1.0), "Every capital-profile split must sum to 100%."
assert all(np.isclose(sum(weights), 1.0) for weights in LAG_WEIGHTS.values()), "Every lag vector must sum to 100%."
display(CATEGORY_SPLITS.map(lambda value: f"{value:.0%}"))

In [ ]:
# 3. Small data-access and time-series helpers
_REFRESHED_DOWNLOADS = set()
_API_TABLE_MEMORY = {}

def _download(url, destination, refresh=False):
    destination = Path(destination)
    destination_key = str(destination.resolve())
    if refresh and destination_key in _REFRESHED_DOWNLOADS:
        return destination
    if destination.exists() and not refresh:
        return destination
    destination.parent.mkdir(parents=True, exist_ok=True)
    if requests is not None:
        response = requests.get(url, timeout=90)
        response.raise_for_status()
        destination.write_bytes(response.content)
    else:
        with urllib.request.urlopen(url, timeout=90) as response:
            destination.write_bytes(response.read())
    _REFRESHED_DOWNLOADS.add(destination_key)
    return destination


def fetch_bea_table_api(dataset, table, refresh=False):
    """Fetch one BEA table using only the BEA_API_KEY environment variable."""
    memory_key = (dataset, table)
    if memory_key in _API_TABLE_MEMORY:
        return _API_TABLE_MEMORY[memory_key].copy()
    cache_path = CACHE_DIR / f"bea_api_{dataset}_{table}.csv"
    if cache_path.exists() and not refresh:
        frame = pd.read_csv(cache_path)
        _API_TABLE_MEMORY[memory_key] = frame
        return frame.copy()
    api_key = os.getenv("BEA_API_KEY", "").strip()
    if not api_key:
        raise RuntimeError("BEA_API_KEY is not set; use the public bulk-workbook fallback.")
    params = {
        "UserID": api_key,
        "method": "GetData",
        "DataSetName": dataset,
        "TableName": table,
        "Frequency": "Q",
        "Year": "ALL",
        "ResultFormat": "JSON",
    }
    url = "https://apps.bea.gov/api/data/?" + urllib.parse.urlencode(params)
    if requests is not None:
        response = requests.get(url, timeout=90)
        response.raise_for_status()
        payload = response.json()
    else:
        with urllib.request.urlopen(url, timeout=90) as response:
            payload = json.loads(response.read().decode("utf-8"))
    rows = payload.get("BEAAPI", {}).get("Results", {}).get("Data")
    if not rows:
        error = payload.get("BEAAPI", {}).get("Results", {}).get("Error", payload)
        raise RuntimeError(f"BEA API returned no data for {dataset}/{table}: {error}")
    frame = pd.DataFrame(rows)
    frame.to_csv(cache_path, index=False)
    _API_TABLE_MEMORY[memory_key] = frame
    return frame.copy()


def _series_from_api(frame, entry):
    match = frame.loc[frame["SeriesCode"].astype(str).eq(entry["code"])].copy()
    if match.empty:
        raise KeyError(f"{entry['code']} not found in BEA API table {entry['table']}")
    label_col = "LineDescription" if "LineDescription" in match.columns else "LineDesc"
    label = str(match[label_col].iloc[0]) if label_col in match.columns else ""
    if entry["label"].lower() not in label.lower():
        raise ValueError(f"Series-label mismatch for {entry['code']}: {label!r}")
    values = pd.to_numeric(match["DataValue"].astype(str).str.replace(",", "", regex=False), errors="coerce")
    index = pd.PeriodIndex(match["TimePeriod"].astype(str), freq="Q")
    return pd.Series(values.to_numpy() * entry["scale"], index=index, name=entry["code"]).sort_index(), label


def _series_from_bulk(entry):
    filename, url = BEA_BULK_FILES[entry["bulk"]]
    path = _download(url, CACHE_DIR / filename, refresh=REFRESH_OFFICIAL_DATA)
    raw = pd.read_excel(path, sheet_name=entry["sheet"], header=None, nrows=7)
    metadata = " | ".join(str(value) for value in raw.iloc[:, 0].dropna().tolist())
    frame = pd.read_excel(path, sheet_name=entry["sheet"], header=7)
    frame = frame.rename(columns={frame.columns[0]: "Line", frame.columns[1]: "Description", frame.columns[2]: "SeriesCode"})
    match = frame.loc[frame["SeriesCode"].astype(str).eq(entry["code"])]
    if match.empty:
        raise KeyError(f"{entry['code']} not found in {path.name}/{entry['sheet']}")
    label = str(match["Description"].iloc[0])
    if entry["label"].lower() not in label.lower():
        raise ValueError(f"Series-label mismatch for {entry['code']}: {label!r}")
    period_columns = [column for column in frame.columns if re.fullmatch(r"\d{4}Q[1-4]", str(column))]
    values = pd.to_numeric(match.iloc[0][period_columns], errors="coerce") * entry["scale"]
    series = pd.Series(values.to_numpy(), index=pd.PeriodIndex(period_columns, freq="Q"), name=entry["code"])
    return series.dropna().sort_index(), label, metadata


def get_bea_series(name):
    entry = SERIES_REGISTRY[name]
    api_key_available = bool(os.getenv("BEA_API_KEY", "").strip())
    api_cache = CACHE_DIR / f"bea_api_{entry['dataset']}_{entry['table']}.csv"
    if api_key_available and (REFRESH_OFFICIAL_DATA or api_cache.exists()):
        frame = fetch_bea_table_api(entry["dataset"], entry["table"], refresh=REFRESH_OFFICIAL_DATA)
        series, label = _series_from_api(frame, entry)
        return series, label, f"BEA API {entry['dataset']}/{entry['table']}"
    series, label, metadata = _series_from_bulk(entry)
    return series, label, metadata


def normalized_seasonality(values):
    values = np.asarray(values, dtype=float)
    values = np.where(np.isfinite(values) & (values >= 0), values, np.nan)
    if np.isfinite(values).sum() == 0 or np.nansum(values) <= 0:
        return np.repeat(0.25, 4)
    values = np.where(np.isfinite(values), values, 0.0)
    return values / values.sum()


def annualized_growth(series):
    return ((series / series.shift(1)) ** 4 - 1.0) * 100.0


def counterfactual_contributions(gdp, investment, imports, periods=4):
    growth = gdp.pct_change(periods) * 100.0
    gross_cf = (gdp - investment).pct_change(periods) * 100.0
    net_cf = (gdp - investment + imports).pct_change(periods) * 100.0
    result = pd.DataFrame({
        "gdp_growth_pct": growth,
        "gross_contribution_pp": growth - gross_cf,
        "net_contribution_pp": growth - net_cf,
    })
    result["import_drag_pp"] = result["net_contribution_pp"] - result["gross_contribution_pp"]
    return result

In [ ]:
# 4. Load and audit the company CAPEX input
REQUIRED_COLUMNS = {"Quarter_End_Date", "Company_Name", "GICS_Industry_Group", "CAPEX"}
if not CSV_PATH.exists():
    raise FileNotFoundError(f"Input file not found: {CSV_PATH}")

capex_raw = pd.read_csv(CSV_PATH)
missing_columns = REQUIRED_COLUMNS.difference(capex_raw.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

capex_raw = capex_raw[list(REQUIRED_COLUMNS)].copy()
capex_raw["Quarter_End_Date"] = pd.to_datetime(capex_raw["Quarter_End_Date"], errors="raise")
capex_raw["quarter"] = capex_raw["Quarter_End_Date"].dt.to_period("Q")
capex_raw["CAPEX"] = pd.to_numeric(capex_raw["CAPEX"], errors="coerce")
capex_raw["Company_Name"] = capex_raw["Company_Name"].astype(str).str.strip()
capex_raw["GICS_Industry_Group"] = capex_raw["GICS_Industry_Group"].astype(str).str.strip()

duplicate_count = int(capex_raw.duplicated(["Company_Name", "quarter"]).sum())
group_conflicts = int((capex_raw.groupby("Company_Name")["GICS_Industry_Group"].nunique(dropna=True) != 1).sum())
blank_industry_rows = int(capex_raw["GICS_Industry_Group"].replace("", np.nan).isna().sum())
total_2024 = capex_raw.loc[capex_raw["quarter"].dt.year.eq(2024), "CAPEX"].sum(min_count=1)

input_audit = pd.DataFrame(
    {
        "Metric": ["Rows", "Companies", "Duplicate company-quarters", "Blank industry rows", "Companies with conflicting industries", "Missing CAPEX values", "2024 CAPEX ($bn)"],
        "Value": [len(capex_raw), capex_raw["Company_Name"].nunique(), duplicate_count, blank_industry_rows, group_conflicts, int(capex_raw["CAPEX"].isna().sum()), total_2024],
    }
)
display(input_audit)

if duplicate_count:
    raise ValueError(f"Found {duplicate_count} duplicate company-quarter keys.")
if blank_industry_rows or group_conflicts:
    raise ValueError("Industry-group validation failed.")
if abs(total_2024 - 490.9) > 1.0:
    warnings.warn(f"2024 CAPEX is ${total_2024:,.1f}bn, outside the expected $490.9bn ± $1bn audit range.")

In [ ]:
# 5. Seasonality and estimate-panel completion
company_meta = capex_raw.groupby("Company_Name", as_index=False).agg(GICS_Industry_Group=("GICS_Industry_Group", "first"))
companies = company_meta["Company_Name"].sort_values().tolist()
all_quarters = pd.period_range(capex_raw["quarter"].min(), ANALYSIS_END, freq="Q")
panel = pd.MultiIndex.from_product([companies, all_quarters], names=["Company_Name", "quarter"]).to_frame(index=False)
panel = panel.merge(company_meta, on="Company_Name", how="left")
panel = panel.merge(capex_raw[["Company_Name", "quarter", "CAPEX"]], on=["Company_Name", "quarter"], how="left")
panel = panel.rename(columns={"CAPEX": "capex_reported"})
panel["year"] = panel["quarter"].dt.year
panel["quarter_num"] = panel["quarter"].dt.quarter

season_source = panel.loc[
    panel["quarter"].le(pd.Period("2025Q4", freq="Q")) & panel["capex_reported"].notna(),
    ["Company_Name", "GICS_Industry_Group", "year", "quarter_num", "capex_reported"],
].copy()
season_source["positive_capex"] = season_source["capex_reported"].clip(lower=0)
year_stats = season_source.groupby(["Company_Name", "year"]).agg(total=("positive_capex", "sum"), count=("positive_capex", "count")).reset_index()
valid_years = year_stats.loc[year_stats["count"].eq(4) & year_stats["total"].gt(0), ["Company_Name", "year", "total"]]
season_source = season_source.merge(valid_years, on=["Company_Name", "year"], how="inner")
season_source["season_share"] = season_source["positive_capex"] / season_source["total"]

company_season = season_source.groupby(["Company_Name", "quarter_num"])["season_share"].median().unstack("quarter_num")
company_season = company_season.reindex(columns=[1, 2, 3, 4])
for company in company_season.index:
    company_season.loc[company] = normalized_seasonality(company_season.loc[company].to_numpy())

company_with_group = company_season.reset_index().merge(company_meta, on="Company_Name", how="left")
industry_season = company_with_group.groupby("GICS_Industry_Group")[[1, 2, 3, 4]].median()
for industry in industry_season.index:
    industry_season.loc[industry] = normalized_seasonality(industry_season.loc[industry].to_numpy())

def get_seasonality(company, industry):
    if company in company_season.index:
        return normalized_seasonality(company_season.loc[company].to_numpy())
    if industry in industry_season.index:
        return normalized_seasonality(industry_season.loc[industry].to_numpy())
    return np.repeat(0.25, 4)

recent_start = LAST_ACTUAL_QUARTER - 3
recent_active = set(panel.loc[panel["quarter"].between(recent_start, LAST_ACTUAL_QUARTER) & panel["capex_reported"].notna(), "Company_Name"])
future_reported = set(panel.loc[panel["quarter"].gt(LAST_ACTUAL_QUARTER) & panel["capex_reported"].notna(), "Company_Name"])
active_companies = recent_active.union(future_reported)

panel["capex_used"] = panel["capex_reported"]
panel["data_status"] = np.select(
    [
        panel["quarter"].le(LAST_ACTUAL_QUARTER) & panel["capex_reported"].notna(),
        panel["quarter"].gt(LAST_ACTUAL_QUARTER) & panel["capex_reported"].notna(),
        panel["quarter"].le(LAST_ACTUAL_QUARTER),
    ],
    ["actual", "reported_estimate", "missing_actual"],
    default="missing_estimate",
)

for company in sorted(active_companies):
    company_index = panel.index[panel["Company_Name"].eq(company)]
    company_frame = panel.loc[company_index].sort_values("quarter")
    series = company_frame.set_index("quarter")["capex_reported"].copy()
    interpolated = series.interpolate(method="linear", limit_area="inside")
    future_missing = series.index.to_series().gt(LAST_ACTUAL_QUARTER).to_numpy() & series.isna().to_numpy()
    internal_fill = future_missing & interpolated.notna().to_numpy()
    filled = series.copy()
    filled.iloc[np.where(internal_fill)[0]] = interpolated.iloc[np.where(internal_fill)[0]]

    remaining_future = filled.index.to_series().gt(LAST_ACTUAL_QUARTER).to_numpy() & filled.isna().to_numpy()
    available = series.dropna()
    if remaining_future.any() and not available.empty:
        latest_four = available.tail(4)
        annual_run_rate = float(latest_four.sum()) if len(latest_four) == 4 else float(latest_four.mean() * 4)
        industry = company_frame["GICS_Industry_Group"].iloc[0]
        season = get_seasonality(company, industry)
        for position in np.where(remaining_future)[0]:
            quarter_number = filled.index[position].quarter
            filled.iloc[position] = max(annual_run_rate * season[quarter_number - 1], 0.0)

    aligned = company_frame[["quarter"]].copy()
    aligned["filled"] = aligned["quarter"].map(filled)
    new_values = aligned["filled"].to_numpy()
    original_values = company_frame["capex_reported"].to_numpy()
    imputed_mask = company_frame["quarter"].gt(LAST_ACTUAL_QUARTER).to_numpy() & pd.isna(original_values) & pd.notna(new_values)
    panel.loc[company_frame.index, "capex_used"] = new_values
    panel.loc[company_frame.index[imputed_mask], "data_status"] = "imputed_estimate"

coverage_audit = panel.groupby("quarter").agg(
    actual_companies=("data_status", lambda s: int(s.eq("actual").sum())),
    reported_estimate_companies=("data_status", lambda s: int(s.eq("reported_estimate").sum())),
    imputed_estimate_companies=("data_status", lambda s: int(s.eq("imputed_estimate").sum())),
    total_capex_bn=("capex_used", "sum"),
)
coverage_audit["reported_capex_bn"] = panel.loc[panel["data_status"].isin(["actual", "reported_estimate"])].groupby("quarter")["capex_used"].sum()
coverage_audit["imputed_capex_bn"] = panel.loc[panel["data_status"].eq("imputed_estimate")].groupby("quarter")["capex_used"].sum()
coverage_audit[["reported_capex_bn", "imputed_capex_bn"]] = coverage_audit[["reported_capex_bn", "imputed_capex_bn"]].fillna(0.0)
coverage_audit["imputed_capex_share"] = coverage_audit["imputed_capex_bn"].div(coverage_audit["total_capex_bn"].replace(0, np.nan))
coverage_audit["warning"] = np.where(coverage_audit["imputed_capex_share"].gt(IMPUTATION_WARNING_SHARE), "REVIEW", "OK")
display(coverage_audit.loc[pd.Period("2025Q1", freq="Q"):ANALYSIS_END])

In [ ]:
# 6. Flat-2022 baseline and capped pre-2022 trend sensitivity
baseline_records = []
first_valid = panel.loc[panel["capex_reported"].notna()].groupby("Company_Name")["quarter"].min()

def annualize_partial(values, quarters, season):
    estimates = []
    for value, quarter_number in zip(values, quarters):
        share = season[int(quarter_number) - 1]
        if pd.notna(value) and value >= 0 and share > 0:
            estimates.append(float(value) / float(share))
    return float(np.median(estimates)) if estimates else np.nan

for row in company_meta.itertuples(index=False):
    company = row.Company_Name
    industry = row.GICS_Industry_Group
    season = get_seasonality(company, industry)
    company_panel = panel.loc[panel["Company_Name"].eq(company)]
    obs_2022 = company_panel.loc[company_panel["year"].eq(2022) & company_panel["capex_reported"].notna()]
    if len(obs_2022) == 4:
        annual_baseline = float(obs_2022["capex_reported"].sum())
        source = "2022_complete"
    elif len(obs_2022) > 0:
        annual_baseline = annualize_partial(obs_2022["capex_reported"], obs_2022["quarter_num"], season)
        source = f"2022_seasonally_annualized_{len(obs_2022)}q"
    elif first_valid.get(company, pd.Period("2099Q4", freq="Q")) > pd.Period("2022Q4", freq="Q"):
        annual_baseline = 0.0
        source = "post_2022_entrant_zero"
    else:
        obs_2021 = company_panel.loc[company_panel["year"].eq(2021) & company_panel["capex_reported"].notna()]
        annual_baseline = annualize_partial(obs_2021["capex_reported"], obs_2021["quarter_num"], season)
        source = "2021_proxy" if pd.notna(annual_baseline) else "missing_baseline"
    baseline_records.append({
        "Company_Name": company,
        "GICS_Industry_Group": industry,
        "baseline_annual_2022_bn": annual_baseline,
        "baseline_source": source,
        "season_q1": season[0], "season_q2": season[1], "season_q3": season[2], "season_q4": season[3],
    })

baseline_audit = pd.DataFrame(baseline_records)
panel = panel.merge(baseline_audit[["Company_Name", "baseline_annual_2022_bn", "baseline_source", "season_q1", "season_q2", "season_q3", "season_q4"]], on="Company_Name", how="left")
season_columns = {1: "season_q1", 2: "season_q2", 3: "season_q3", 4: "season_q4"}
panel["season_share"] = [panel.at[i, season_columns[int(q)]] for i, q in zip(panel.index, panel["quarter_num"])]
panel["baseline_flat_bn"] = panel["baseline_annual_2022_bn"] * panel["season_share"]
panel.loc[panel["capex_used"].isna(), "baseline_flat_bn"] = np.nan
panel["raw_excess_bn"] = panel["capex_used"] - panel["baseline_flat_bn"]
panel["gross_ai_capex_bn"] = panel["raw_excess_bn"].clip(lower=0)
panel["unfloored_net_wedge_bn"] = panel["raw_excess_bn"]

trend_records = []
annual_history = panel.loc[panel["year"].between(2017, 2021) & panel["capex_reported"].notna()].groupby(["Company_Name", "year"]).agg(total=("capex_reported", "sum"), count=("capex_reported", "count")).reset_index()
annual_history["annualized_total"] = annual_history["total"] * 4.0 / annual_history["count"]
for row in baseline_audit.itertuples(index=False):
    history = annual_history.loc[(annual_history["Company_Name"].eq(row.Company_Name)) & annual_history["annualized_total"].gt(0)]
    if len(history) >= 3:
        slope, intercept = np.polyfit(history["year"].to_numpy(dtype=float), np.log(history["annualized_total"].to_numpy(dtype=float)), 1)
        growth = float(np.clip(np.exp(slope) - 1.0, *TREND_GROWTH_ANNUAL_BOUNDS))
        fitted_2021 = float(np.exp(intercept + slope * 2021))
        trend_source = "2017_2021_log_linear_capped"
    else:
        growth = 0.0
        fitted_2021 = float(row.baseline_annual_2022_bn) if pd.notna(row.baseline_annual_2022_bn) else np.nan
        trend_source = "flat_fallback"
    trend_records.append({"Company_Name": row.Company_Name, "trend_fitted_2021_bn": fitted_2021, "trend_growth": growth, "trend_source": trend_source})

trend_audit = pd.DataFrame(trend_records)
baseline_audit = baseline_audit.merge(trend_audit, on="Company_Name", how="left")
panel = panel.merge(trend_audit, on="Company_Name", how="left")
panel["trend_baseline_annual_bn"] = panel["trend_fitted_2021_bn"] * (1.0 + panel["trend_growth"]) ** (panel["year"] - 2021)
panel["baseline_trend_bn"] = panel["trend_baseline_annual_bn"] * panel["season_share"]
panel.loc[panel["capex_used"].isna(), "baseline_trend_bn"] = np.nan
panel["trend_raw_excess_bn"] = panel["capex_used"] - panel["baseline_trend_bn"]
panel["trend_gross_ai_capex_bn"] = panel["trend_raw_excess_bn"].clip(lower=0)

complete_2022_companies = set(baseline_audit.loc[baseline_audit["baseline_source"].eq("2022_complete"), "Company_Name"])
complete_2022_excess = panel.loc[panel["Company_Name"].isin(complete_2022_companies) & panel["year"].eq(2022), "raw_excess_bn"].sum()
print(f"2022 unfloored excess for complete-2022 cohort: ${complete_2022_excess:,.10f}bn")
display(baseline_audit["baseline_source"].value_counts().rename_axis("baseline_source").to_frame("companies"))

In [ ]:
# 7. US allocation, GDP asset mapping, and recognition timing
panel["capital_profile"] = panel["Company_Name"].map(PROFILE_OVERRIDES)
panel["capital_profile"] = panel["capital_profile"].fillna(panel["GICS_Industry_Group"].map(INDUSTRY_PROFILE))
if panel["capital_profile"].isna().any():
    missing_profiles = panel.loc[panel["capital_profile"].isna(), "GICS_Industry_Group"].unique().tolist()
    raise ValueError(f"Missing capital-profile mappings for: {missing_profiles}")
panel["us_share"] = np.where(panel["capital_profile"].eq("utilities"), US_SHARE_UTILITY, US_SHARE_DEFAULT)
panel["us_share"] = panel["Company_Name"].map(COMPANY_US_SHARE_OVERRIDES).fillna(panel["us_share"])

INVESTMENT_CATEGORIES = [column for column in CATEGORY_SPLITS.columns if column != "land_excluded"]

def allocate_and_time(source_panel, capex_column, scenario_name):
    model_panel = source_panel.loc[source_panel["quarter"].between(MODEL_START, ANALYSIS_END)].copy()
    model_panel["us_ai_source_bn"] = model_panel[capex_column] * model_panel["us_share"]
    split_long = []
    for category in CATEGORY_SPLITS.columns:
        block = model_panel[["Company_Name", "GICS_Industry_Group", "capital_profile", "quarter", "us_ai_source_bn"]].copy()
        block["category"] = category
        block["split"] = block["capital_profile"].map(CATEGORY_SPLITS[category])
        block["source_amount_bn"] = block["us_ai_source_bn"] * block["split"]
        split_long.append(block)
    allocated = pd.concat(split_long, ignore_index=True)
    allocated["scenario"] = scenario_name
    land = allocated.loc[allocated["category"].eq("land_excluded")].copy()

    timed_blocks = []
    for category, weights in LAG_WEIGHTS.items():
        category_block = allocated.loc[allocated["category"].eq(category)].copy()
        for lag, weight in enumerate(weights):
            lagged = category_block.copy()
            lagged["recorded_quarter"] = lagged["quarter"] + lag
            lagged["lag"] = lag
            lagged["lag_weight"] = weight
            lagged["recorded_amount_bn"] = lagged["source_amount_bn"] * weight
            timed_blocks.append(lagged)
    timed = pd.concat(timed_blocks, ignore_index=True)
    timed = timed.loc[timed["recorded_quarter"].le(TAIL_END)].copy()
    return allocated, timed, land

allocated_flat, timed_flat, land_flat = allocate_and_time(panel, "gross_ai_capex_bn", "flat_2022")
allocated_trend, timed_trend, land_trend = allocate_and_time(panel, "trend_gross_ai_capex_bn", "capped_trend")

eligible_source_flat = allocated_flat.loc[allocated_flat["category"].isin(INVESTMENT_CATEGORIES), "source_amount_bn"].sum()
recorded_flat = timed_flat["recorded_amount_bn"].sum()
timing_conservation_difference = recorded_flat - eligible_source_flat
us_source_flat = allocated_flat.drop_duplicates(["Company_Name", "quarter"])["us_ai_source_bn"].sum()
land_source_flat = land_flat["source_amount_bn"].sum()
allocation_identity_difference = eligible_source_flat + land_source_flat - us_source_flat

print(f"Allocation identity difference: ${allocation_identity_difference:,.10f}bn")
print(f"Timing conservation difference through {TAIL_END}: ${timing_conservation_difference:,.10f}bn")

In [ ]:
# 8. Official BEA and Census data, with label validation and cache metadata
official_series = {}
official_sources = []
for name in SERIES_REGISTRY:
    series, label, metadata = get_bea_series(name)
    official_series[name] = series
    official_sources.append({
        "series_name": name,
        "series_code": SERIES_REGISTRY[name]["code"],
        "label": label,
        "dataset_table": f"{SERIES_REGISTRY[name]['dataset']}/{SERIES_REGISTRY[name]['table']}",
        "source_or_vintage": metadata,
    })

sources_audit = pd.DataFrame(official_sources)
nominal_meta = sources_audit.loc[sources_audit["series_name"].eq("nominal_gdp"), "source_or_vintage"].iloc[0]
vintage_match = re.search(r"Data published [^|]+", nominal_meta)
OFFICIAL_DATA_VINTAGE = vintage_match.group(0).strip() if vintage_match else nominal_meta[:120]

# Data-center price indexes begin in 2020. Ratio-splice the office index before that date.
dc_price_raw = official_series["dc_structures_price"]
office_price = official_series["office_price"]
first_dc_period = dc_price_raw.first_valid_index()
splice_ratio = dc_price_raw.loc[first_dc_period] / office_price.loc[first_dc_period]
dc_price_spliced = dc_price_raw.combine_first(office_price * splice_ratio).sort_index()
official_series["dc_structures_price_spliced"] = dc_price_spliced
official_series["power_cooling_price"] = pd.concat(
    [official_series["communication_equipment_price"], official_series["electrical_equipment_price"]], axis=1
).mean(axis=1)

# Census data centers are a cross-check only.
census_dc_quarterly = pd.Series(dtype=float, name="census_dc_saar_bn")
try:
    census_filename, census_url = BEA_BULK_FILES["census"]
    census_path = _download(census_url, CACHE_DIR / census_filename, refresh=REFRESH_OFFICIAL_DATA)
    census = pd.read_excel(census_path, sheet_name="Private SA", header=3, usecols=[0, 9])
    census.columns = ["date_text", "data_center_millions_saar"]
    census["month"] = pd.to_datetime(census["date_text"].astype(str).str.replace(r"[pr]$", "", regex=True), format="%b-%y", errors="coerce")
    census["quarter"] = census["month"].dt.to_period("Q")
    census["data_center_millions_saar"] = pd.to_numeric(census["data_center_millions_saar"], errors="coerce")
    census_dc_quarterly = (census.dropna(subset=["quarter"]).groupby("quarter")["data_center_millions_saar"].mean() / 1000.0).sort_index()
    census_dc_quarterly.name = "census_dc_saar_bn"
except Exception as exc:
    warnings.warn(f"Census C30 cross-check unavailable: {exc}")

print(OFFICIAL_DATA_VINTAGE)
display(sources_audit[["series_name", "series_code", "label", "dataset_table"]])

In [ ]:
# 9. Extend GDP and category deflators through the recognition tail
projection_index = pd.period_range(MODEL_START, TAIL_END, freq="Q")

def extend_price_index(series, target_index, annual_bounds=COMPONENT_PRICE_ANNUAL_BOUNDS):
    result = series.reindex(target_index).copy()
    observed = result.dropna()
    if observed.empty:
        raise ValueError("Cannot project an empty price-index series.")
    recent_log_changes = np.log(observed / observed.shift(1)).dropna().tail(12)
    annual_rate = float(np.expm1(recent_log_changes.median() * 4)) if not recent_log_changes.empty else 0.0
    annual_rate = float(np.clip(annual_rate, *annual_bounds))
    quarterly_rate = (1.0 + annual_rate) ** 0.25 - 1.0
    last_period = observed.index.max()
    last_value = float(observed.loc[last_period])
    for quarter in target_index[target_index > last_period]:
        last_value *= 1.0 + quarterly_rate
        result.loc[quarter] = last_value
    return result, annual_rate

nominal_gdp = official_series["nominal_gdp"].reindex(projection_index)
real_gdp = official_series["real_gdp"].reindex(projection_index)
gdp_deflator_observed = (official_series["nominal_gdp"] / official_series["real_gdp"] * 100.0).dropna()
gdp_inflation_history = gdp_deflator_observed.pct_change(4).dropna().tail(12)
gdp_inflation_annual = float(np.clip(gdp_inflation_history.median(), *GDP_INFLATION_ANNUAL_BOUNDS))
nominal_gdp_quarterly_growth = (1.0 + NOMINAL_GDP_GROWTH_ANNUAL) ** 0.25 - 1.0
real_gdp_quarterly_growth = (1.0 + nominal_gdp_quarterly_growth) / ((1.0 + gdp_inflation_annual) ** 0.25) - 1.0
latest_gdp_period = official_series["nominal_gdp"].dropna().index.max()
for quarter in projection_index[projection_index > latest_gdp_period]:
    nominal_gdp.loc[quarter] = nominal_gdp.loc[quarter - 1] * (1.0 + nominal_gdp_quarterly_growth)
    real_gdp.loc[quarter] = real_gdp.loc[quarter - 1] * (1.0 + real_gdp_quarterly_growth)

category_price_sources = {
    "dc_structures": official_series["dc_structures_price_spliced"],
    "power_structures": official_series["power_structures_price"],
    "it_equipment": official_series["computers_price"],
    "power_cooling_equipment": official_series["power_cooling_price"],
    "other_pfi": official_series["other_equipment_price"],
}
category_prices = {}
price_projection_rates = {}
for category, source in category_price_sources.items():
    category_prices[category], price_projection_rates[category] = extend_price_index(source, projection_index)
computer_import_price, price_projection_rates["computer_imports"] = extend_price_index(official_series["computer_imports_price"], projection_index)

projection_assumptions = pd.DataFrame(
    {
        "Assumption": ["Nominal GDP annual growth", "GDP inflation annual", "Implied real GDP annual growth"] + [f"{key} price annual growth" for key in price_projection_rates],
        "Value": [NOMINAL_GDP_GROWTH_ANNUAL, gdp_inflation_annual, (1.0 + real_gdp_quarterly_growth) ** 4 - 1.0] + list(price_projection_rates.values()),
    }
)
projection_assumptions_display = projection_assumptions.copy()
projection_assumptions_display["Value"] = projection_assumptions_display["Value"].map(lambda value: f"{value:.2%}")
display(projection_assumptions_display)

In [ ]:
# 10. Separate investment/import flows, deflate them, and calculate contribution scenarios
def build_flow_scenario(timed_detail, import_case, scenario_name):
    flows = timed_detail.groupby(["recorded_quarter", "category"], as_index=False)["recorded_amount_bn"].sum()
    flows = flows.rename(columns={"recorded_quarter": "quarter", "recorded_amount_bn": "gross_investment_nominal_bn"})
    flows["import_share"] = flows["category"].map(IMPORT_SHARES[import_case])
    flows["imports_nominal_bn"] = flows["gross_investment_nominal_bn"] * flows["import_share"]
    flows["investment_price_index"] = [category_prices[category].get(quarter, np.nan) for category, quarter in zip(flows["category"], flows["quarter"])]
    flows["import_price_index"] = flows["investment_price_index"]
    computer_rows = flows["category"].eq("it_equipment")
    flows.loc[computer_rows, "import_price_index"] = [computer_import_price.get(quarter, np.nan) for quarter in flows.loc[computer_rows, "quarter"]]
    flows["gross_investment_real_bn"] = flows["gross_investment_nominal_bn"] / (flows["investment_price_index"] / 100.0)
    flows["imports_real_bn"] = flows["imports_nominal_bn"] / (flows["import_price_index"] / 100.0)
    flows["scenario"] = scenario_name
    flows["import_case"] = import_case

    quarterly = flows.groupby("quarter").agg(
        gross_investment_nominal_bn=("gross_investment_nominal_bn", "sum"),
        imports_nominal_bn=("imports_nominal_bn", "sum"),
        gross_investment_real_bn=("gross_investment_real_bn", "sum"),
        imports_real_bn=("imports_real_bn", "sum"),
    ).reindex(projection_index, fill_value=0.0)
    quarterly["gross_investment_nominal_saar_bn"] = quarterly["gross_investment_nominal_bn"] * 4.0
    quarterly["imports_nominal_saar_bn"] = quarterly["imports_nominal_bn"] * 4.0
    quarterly["net_nominal_saar_bn"] = quarterly["gross_investment_nominal_saar_bn"] - quarterly["imports_nominal_saar_bn"]
    quarterly["gross_investment_real_saar_bn"] = quarterly["gross_investment_real_bn"] * 4.0
    quarterly["imports_real_saar_bn"] = quarterly["imports_real_bn"] * 4.0
    quarterly["net_real_saar_bn"] = quarterly["gross_investment_real_saar_bn"] - quarterly["imports_real_saar_bn"]
    quarterly["scenario"] = scenario_name
    quarterly["import_case"] = import_case
    return flows, quarterly

scenario_inputs = {
    "flat_low": (timed_flat, "low"),
    "flat_base": (timed_flat, "base"),
    "flat_high": (timed_flat, "high"),
    "trend_base": (timed_trend, "base"),
}
asset_flow_scenarios = {}
quarterly_flow_scenarios = {}
quarterly_contribution_scenarios = {}
annual_contribution_scenarios = {}

for scenario_name, (timed_detail, import_case) in scenario_inputs.items():
    asset_flows, quarterly_flows = build_flow_scenario(timed_detail, import_case, scenario_name)
    asset_flow_scenarios[scenario_name] = asset_flows
    quarterly_flow_scenarios[scenario_name] = quarterly_flows

    nominal_contrib = counterfactual_contributions(
        nominal_gdp,
        quarterly_flows["gross_investment_nominal_saar_bn"],
        quarterly_flows["imports_nominal_saar_bn"],
        periods=4,
    ).add_prefix("nominal_")
    real_contrib = counterfactual_contributions(
        real_gdp,
        quarterly_flows["gross_investment_real_saar_bn"],
        quarterly_flows["imports_real_saar_bn"],
        periods=4,
    ).add_prefix("real_")
    quarterly_contribution_scenarios[scenario_name] = pd.concat([quarterly_flows, nominal_contrib, real_contrib], axis=1)

    annual = pd.DataFrame(index=range(MODEL_START.year, TAIL_END.year + 1))
    annual.index.name = "year"
    annual["nominal_gdp_bn"] = nominal_gdp.groupby(nominal_gdp.index.year).mean()
    annual["real_gdp_2017_bn"] = real_gdp.groupby(real_gdp.index.year).mean()
    for column in ["gross_investment_nominal_saar_bn", "imports_nominal_saar_bn", "net_nominal_saar_bn", "gross_investment_real_saar_bn", "imports_real_saar_bn", "net_real_saar_bn"]:
        annual[column] = quarterly_flows[column].groupby(quarterly_flows.index.year).mean()
    nominal_annual = counterfactual_contributions(annual["nominal_gdp_bn"], annual["gross_investment_nominal_saar_bn"], annual["imports_nominal_saar_bn"], periods=1).add_prefix("nominal_")
    real_annual = counterfactual_contributions(annual["real_gdp_2017_bn"], annual["gross_investment_real_saar_bn"], annual["imports_real_saar_bn"], periods=1).add_prefix("real_")
    annual = pd.concat([annual, nominal_annual, real_annual], axis=1)
    annual["gross_investment_share_gdp_pct"] = annual["gross_investment_nominal_saar_bn"] / annual["nominal_gdp_bn"] * 100.0
    annual["net_measured_share_gdp_pct"] = annual["net_nominal_saar_bn"] / annual["nominal_gdp_bn"] * 100.0
    annual["scenario"] = scenario_name
    annual_contribution_scenarios[scenario_name] = annual

central_quarterly = quarterly_contribution_scenarios["flat_base"]
central_annual = annual_contribution_scenarios["flat_base"].loc[2023:2028].copy()
display(central_annual)

In [ ]:
# 11. Headline tables, sensitivities, company/industry detail, and external diagnostics
panel_output = panel.loc[panel["quarter"].between(MODEL_START, ANALYSIS_END)].copy()
panel_output["us_ai_capex_source_bn"] = panel_output["gross_ai_capex_bn"] * panel_output["us_share"]
panel_output["us_unfloored_wedge_bn"] = panel_output["unfloored_net_wedge_bn"] * panel_output["us_share"]

source_annual = panel_output.groupby("year").agg(
    reported_and_imputed_capex_bn=("capex_used", "sum"),
    flat_baseline_bn=("baseline_flat_bn", "sum"),
    gross_ai_capex_bn=("gross_ai_capex_bn", "sum"),
    unfloored_net_wedge_bn=("unfloored_net_wedge_bn", "sum"),
    us_ai_capex_source_bn=("us_ai_capex_source_bn", "sum"),
    us_unfloored_wedge_bn=("us_unfloored_wedge_bn", "sum"),
)
source_annual["flooring_uplift_bn"] = source_annual["gross_ai_capex_bn"] - source_annual["unfloored_net_wedge_bn"]

annual_imputed = panel_output.loc[panel_output["data_status"].eq("imputed_estimate")].groupby("year")["capex_used"].sum()
source_annual["imputed_capex_bn"] = annual_imputed
source_annual["imputed_capex_share"] = source_annual["imputed_capex_bn"].fillna(0.0) / source_annual["reported_and_imputed_capex_bn"]

annual_summary = central_annual.join(source_annual, how="left")
annual_summary["data_status"] = np.select(
    [annual_summary.index <= 2025, annual_summary.index == 2026],
    ["Actual", "Actual through Q2 / estimate thereafter"],
    default="Estimate",
)
annual_summary["gs_us_ai_capex_anchor_bn"] = np.where(annual_summary.index == 2026, GS_US_AI_CAPEX_2026_BN, np.nan)
annual_summary["gs_share_anchor_pct"] = annual_summary.index.map(GS_AI_CAPEX_SHARE_ANCHORS)

baseline_sensitivity = pd.concat(
    {
        "flat_2022": annual_contribution_scenarios["flat_base"].loc[2023:2028],
        "capped_trend": annual_contribution_scenarios["trend_base"].loc[2023:2028],
    }, names=["baseline", "year"]
).reset_index()
baseline_sensitivity = baseline_sensitivity[[
    "baseline", "year", "gross_investment_nominal_saar_bn", "net_nominal_saar_bn",
    "nominal_net_contribution_pp", "real_net_contribution_pp", "gross_investment_share_gdp_pct",
]]

import_sensitivity = pd.concat(
    {case: annual_contribution_scenarios[f"flat_{case}"].loc[2023:2028] for case in ["low", "base", "high"]},
    names=["import_case", "year"],
).reset_index()
import_sensitivity = import_sensitivity[[
    "import_case", "year", "imports_nominal_saar_bn", "net_nominal_saar_bn",
    "nominal_net_contribution_pp", "real_net_contribution_pp", "net_measured_share_gdp_pct",
]]

industry_breakdown = panel_output.groupby(["year", "GICS_Industry_Group"]).agg(
    gross_ai_capex_bn=("gross_ai_capex_bn", "sum"),
    us_ai_capex_source_bn=("us_ai_capex_source_bn", "sum"),
    unfloored_net_wedge_bn=("unfloored_net_wedge_bn", "sum"),
).reset_index()

asset_breakdown = asset_flow_scenarios["flat_base"].groupby([asset_flow_scenarios["flat_base"]["quarter"].dt.year, "category"]).agg(
    gross_investment_nominal_bn=("gross_investment_nominal_bn", "sum"),
    imports_nominal_bn=("imports_nominal_bn", "sum"),
    gross_investment_real_bn=("gross_investment_real_bn", "sum"),
    imports_real_bn=("imports_real_bn", "sum"),
).reset_index().rename(columns={"quarter": "year"})

# Diagnostic: additive approximation versus BEA's published computer contribution to real PFI.
real_pfi = official_series["real_pfi"]
computers_real = official_series["computers_real"]
common = real_pfi.index.intersection(computers_real.index)
pfi_growth = annualized_growth(real_pfi.loc[common])
pfi_without_computers_growth = annualized_growth(real_pfi.loc[common] - computers_real.loc[common])
computer_additive_contribution = pfi_growth - pfi_without_computers_growth
published_computer_contribution = official_series["computer_pfi_contribution"].reindex(common)
diagnostic_compare = pd.concat(
    [computer_additive_contribution.rename("additive_approximation_pp"), published_computer_contribution.rename("bea_fisher_contribution_pp")], axis=1
).dropna()
diagnostic_compare["error_pp"] = diagnostic_compare["additive_approximation_pp"] - diagnostic_compare["bea_fisher_contribution_pp"]
diagnostic_recent = diagnostic_compare.loc[pd.Period("2017Q1", freq="Q"):]

cross_checks = pd.DataFrame(
    {
        "Check": [
            "Bottom-up US AI source capex, 2026",
            "Goldman US AI investment anchor, 2026",
            "Bottom-up gross recognized PFI share of GDP, 2026",
            "Goldman AI capex share anchor, 2026",
            "BEA computer-contribution diagnostic median absolute error since 2017",
            "BEA computer-contribution diagnostic correlation since 2017",
            "Census data-center construction, latest quarter",
            "BEA data-center structures, latest quarter",
        ],
        "Value": [
            annual_summary.loc[2026, "us_ai_capex_source_bn"],
            GS_US_AI_CAPEX_2026_BN,
            annual_summary.loc[2026, "gross_investment_share_gdp_pct"],
            GS_AI_CAPEX_SHARE_ANCHORS[2026],
            diagnostic_recent["error_pp"].abs().median(),
            diagnostic_recent[["additive_approximation_pp", "bea_fisher_contribution_pp"]].corr().iloc[0, 1],
            census_dc_quarterly.dropna().iloc[-1] if not census_dc_quarterly.dropna().empty else np.nan,
            official_series["dc_structures_nominal"].dropna().iloc[-1] if not official_series["dc_structures_nominal"].dropna().empty else np.nan,
        ],
        "Units": ["$bn", "$bn", "% GDP", "% GDP", "percentage points", "correlation", "$bn SAAR", "$bn SAAR"],
        "Interpretation": [
            "CSV perimeter; not calibrated to Goldman",
            "Broader external diagnostic including private/other projects",
            "GDP-recorded eligible investment after timing and land exclusion",
            "External diagnostic only",
            "Chain-dollar additive approximation is not expected to reproduce Fisher contributions exactly",
            "Direction/scale diagnostic only",
            "Census C30 cross-check",
            "BEA underlying-detail cross-check",
        ],
    }
)

display(annual_summary[[
    "data_status", "reported_and_imputed_capex_bn", "flat_baseline_bn", "gross_ai_capex_bn",
    "us_ai_capex_source_bn", "gross_investment_nominal_saar_bn", "imports_nominal_saar_bn",
    "net_nominal_saar_bn", "nominal_net_contribution_pp", "real_net_contribution_pp",
    "gross_investment_share_gdp_pct", "imputed_capex_share",
]])
display(cross_checks)

In [ ]:
# 12. Model checks
central_assets = asset_flow_scenarios["flat_base"]
max_import_excess = max(0.0, float((central_assets["imports_nominal_bn"] - central_assets["gross_investment_nominal_bn"]).max()))
net_identity_difference = float((central_quarterly["net_nominal_saar_bn"] - (central_quarterly["gross_investment_nominal_saar_bn"] - central_quarterly["imports_nominal_saar_bn"])).abs().max())
future_zero_fill_count = int(((panel["quarter"].gt(LAST_ACTUAL_QUARTER)) & panel["capex_reported"].isna() & panel["capex_used"].eq(0) & ~panel["data_status"].eq("imputed_estimate")).sum())

synthetic_gdp = pd.Series([100.0, 104.0, 108.0, 112.0, 116.0], index=pd.period_range("2022Q1", periods=5, freq="Q"))
synthetic_investment = pd.Series([5.0, 6.0, 7.0, 8.0, 9.0], index=synthetic_gdp.index)
synthetic_imports = pd.Series([1.0, 1.2, 1.4, 1.6, 1.8], index=synthetic_gdp.index)
synthetic_nominal = counterfactual_contributions(synthetic_gdp, synthetic_investment, synthetic_imports, periods=4)
synthetic_real = counterfactual_contributions(synthetic_gdp, synthetic_investment, synthetic_imports, periods=4)
synthetic_difference = float((synthetic_nominal - synthetic_real).abs().max().max())

check_rows = [
    ("Input row count", len(capex_raw), 22339, len(capex_raw) - 22339, 0, len(capex_raw) == 22339, "Current source-file audit"),
    ("Unique companies", capex_raw["Company_Name"].nunique(), 769, capex_raw["Company_Name"].nunique() - 769, 0, capex_raw["Company_Name"].nunique() == 769, "Current source-file audit"),
    ("Duplicate company-quarter keys", duplicate_count, 0, duplicate_count, 0, duplicate_count == 0, "Must be zero"),
    ("Blank industry rows", blank_industry_rows, 0, blank_industry_rows, 0, blank_industry_rows == 0, "Must be zero"),
    ("2024 CAPEX tie-out", total_2024, 490.9, total_2024 - 490.9, 1.0, abs(total_2024 - 490.9) <= 1.0, "$bn"),
    ("Capital-profile splits", float((CATEGORY_SPLITS.sum(axis=1) - 1.0).abs().max()), 0, float((CATEGORY_SPLITS.sum(axis=1) - 1.0).abs().max()), 1e-12, np.allclose(CATEGORY_SPLITS.sum(axis=1), 1.0), "Maximum absolute row error"),
    ("Lag weights", max(abs(sum(v) - 1.0) for v in LAG_WEIGHTS.values()), 0, max(abs(sum(v) - 1.0) for v in LAG_WEIGHTS.values()), 1e-12, all(np.isclose(sum(v), 1.0) for v in LAG_WEIGHTS.values()), "Maximum absolute vector error"),
    ("2022 baseline, complete cohort", complete_2022_excess, 0, complete_2022_excess, 1e-8, abs(complete_2022_excess) <= 1e-8, "$bn unfloored excess"),
    ("US allocation identity", allocation_identity_difference, 0, allocation_identity_difference, 1e-8, abs(allocation_identity_difference) <= 1e-8, "$bn"),
    ("Timing conservation through tail", timing_conservation_difference, 0, timing_conservation_difference, 1e-8, abs(timing_conservation_difference) <= 1e-8, "$bn"),
    ("Imports do not exceed investment", max_import_excess, 0, max_import_excess, 1e-12, max_import_excess <= 1e-12, "$bn"),
    ("Net wedge identity", net_identity_difference, 0, net_identity_difference, 1e-9, net_identity_difference <= 1e-9, "$bn SAAR"),
    ("Constant-price nominal/real synthetic test", synthetic_difference, 0, synthetic_difference, 1e-12, synthetic_difference <= 1e-12, "Percentage points"),
    ("Unlabeled forecast zeros", future_zero_fill_count, 0, future_zero_fill_count, 0, future_zero_fill_count == 0, "No missing estimate is silently converted to zero"),
]
checks = pd.DataFrame(check_rows, columns=["Check", "Actual", "Expected", "Difference", "Tolerance", "Passed", "Notes"])
checks["Status"] = np.where(checks["Passed"], "OK", "REVIEW")
checks = checks[["Check", "Actual", "Expected", "Difference", "Tolerance", "Status", "Notes"]]
display(checks)
if checks["Status"].eq("REVIEW").any():
    warnings.warn("One or more model checks require review; inspect the Checks table before using headline results.")

In [ ]:
# 13. Charts — matplotlib when installed; Pillow fallback otherwise
chart_paths = [
    CHART_DIR / "01_real_gdp_contribution.png",
    CHART_DIR / "02_capex_baseline_excess.png",
    CHART_DIR / "03_industry_asset_breakdown.png",
    CHART_DIR / "04_gdp_share.png",
]

def shade_estimates(ax):
    ax.axvspan(2026.5, 2028.5, color="#D9E2F3", alpha=0.45, label="Estimate")

def save_pillow_line_chart(title, frame, path, ylabel, label_map=None):
    from PIL import Image, ImageDraw, ImageFont
    image = Image.new("RGB", (1400, 800), "white")
    draw = ImageDraw.Draw(image)
    title_font = ImageFont.load_default(size=28)
    body_font = ImageFont.load_default(size=18)
    draw.text((70, 35), title, fill="#17365D", font=title_font)
    left, top, right, bottom = 100, 120, 1320, 700
    draw.line((left, bottom, right, bottom), fill="#808080", width=2)
    draw.line((left, top, left, bottom), fill="#808080", width=2)
    numeric = frame.select_dtypes(include=[np.number]).replace([np.inf, -np.inf], np.nan)
    values = numeric.to_numpy(dtype=float)
    finite = values[np.isfinite(values)]
    ymin = min(0.0, float(finite.min())) if finite.size else 0.0
    ymax = max(1.0, float(finite.max())) if finite.size else 1.0
    if ymax == ymin:
        ymax = ymin + 1.0
    colors = ["#4472C4", "#ED7D31", "#70AD47", "#A5A5A5", "#FFC000", "#5B9BD5"]
    label_map = label_map or {}
    xlabels = [str(x) for x in numeric.index]
    for j, column in enumerate(numeric.columns):
        points = []
        for i, value in enumerate(numeric[column]):
            if not np.isfinite(value):
                continue
            x = left + (right - left) * (i / max(len(numeric) - 1, 1))
            y = bottom - (bottom - top) * ((float(value) - ymin) / (ymax - ymin))
            points.append((x, y))
        if len(points) > 1:
            draw.line(points, fill=colors[j % len(colors)], width=4)
        for x, y in points:
            draw.ellipse((x - 4, y - 4, x + 4, y + 4), fill=colors[j % len(colors)])
        label = label_map.get(column, str(column).replace("_", " ").title())
        draw.text((880, 45 + j * 24), label, fill=colors[j % len(colors)], font=body_font)
    for tick in range(6):
        tick_value = ymin + (ymax - ymin) * tick / 5
        tick_y = bottom - (bottom - top) * tick / 5
        draw.line((left - 5, tick_y, right, tick_y), fill="#E7E6E6", width=1)
        draw.text((left - 85, tick_y - 8), f"{tick_value:,.2f}", fill="#666666", font=body_font)
    for i, label in enumerate(xlabels):
        x = left + (right - left) * (i / max(len(xlabels) - 1, 1))
        box = draw.textbbox((0, 0), label, font=body_font)
        text_width = box[2] - box[0]
        text_x = max(5, min(image.width - text_width - 5, x - text_width / 2))
        draw.text((text_x, bottom + 18), label, fill="#404040", font=body_font)
    draw.text((15, top - 30), ylabel, fill="#404040", font=body_font)
    image.save(path)

def ensure_fallback_year_labels(path, year_values):
    """Guarantee year labels in minimal Pillow environments."""
    from PIL import Image, ImageDraw, ImageFont
    image = Image.open(path).convert("RGB")
    draw = ImageDraw.Draw(image)
    font = ImageFont.load_default(size=18)
    left, right, y = 100, 1320, 718
    labels = [str(value) for value in year_values]
    for index, label in enumerate(labels):
        x = left + (right - left) * (index / max(len(labels) - 1, 1))
        box = draw.textbbox((0, 0), label, font=font)
        draw.text((x - (box[2] - box[0]) / 2, y), label, fill="#404040", font=font)
    image.save(path)

years = annual_summary.index
if HAS_MATPLOTLIB:
    plt.style.use("seaborn-v0_8-whitegrid")

    fig, ax = plt.subplots(figsize=(11, 6))
    x = np.arange(len(years))
    ax.bar(x - 0.22, annual_summary["real_gross_contribution_pp"], width=0.44, label="Gross investment", color="#4472C4")
    ax.bar(x + 0.22, annual_summary["real_import_drag_pp"], width=0.44, label="Import drag", color="#ED7D31")
    ax.plot(x, annual_summary["real_net_contribution_pp"], color="#17365D", marker="o", linewidth=2.5, label="Net measured")
    ax.set_xticks(x, years)
    ax.set_ylabel("Percentage points of real GDP growth")
    ax.set_title("AI investment: gross contribution, import drag, and net measured contribution")
    ax.axvspan(3.5, 5.5, color="#D9E2F3", alpha=0.45)
    ax.legend(frameon=False)
    fig.tight_layout(); fig.savefig(chart_paths[0], dpi=170); plt.close(fig)

    fig, ax = plt.subplots(figsize=(11, 6))
    ax.plot(years, annual_summary["reported_and_imputed_capex_bn"], marker="o", label="Company CAPEX", color="#17365D")
    ax.plot(years, annual_summary["flat_baseline_bn"], marker="o", label="Flat-2022 baseline", color="#A5A5A5")
    ax.fill_between(years, 0, annual_summary["gross_ai_capex_bn"], alpha=0.35, label="Gross AI excess", color="#70AD47")
    shade_estimates(ax)
    ax.set_ylabel("$bn, calendar-year total")
    ax.set_title("Company CAPEX, counterfactual baseline, and gross AI-related excess")
    ax.legend(frameon=False)
    fig.tight_layout(); fig.savefig(chart_paths[1], dpi=170); plt.close(fig)

    industry_pivot = industry_breakdown.loc[industry_breakdown["year"].between(2023, 2028)].pivot(index="year", columns="GICS_Industry_Group", values="us_ai_capex_source_bn").fillna(0)
    asset_pivot = asset_breakdown.loc[asset_breakdown["year"].between(2023, 2028)].pivot(index="year", columns="category", values="gross_investment_nominal_bn").fillna(0)
    fig, axes = plt.subplots(1, 2, figsize=(15, 6), sharex=False)
    industry_pivot.plot(kind="bar", stacked=True, ax=axes[0], colormap="tab20")
    asset_pivot.plot(kind="bar", stacked=True, ax=axes[1], colormap="Set2")
    axes[0].set_title("US AI CAPEX by industry"); axes[0].set_ylabel("$bn")
    axes[1].set_title("GDP-recorded AI investment by asset"); axes[1].set_ylabel("$bn")
    for ax in axes: ax.legend(frameon=False, fontsize=8)
    fig.tight_layout(); fig.savefig(chart_paths[2], dpi=170); plt.close(fig)

    fig, ax = plt.subplots(figsize=(11, 6))
    ax.plot(years, annual_summary["gross_investment_share_gdp_pct"], marker="o", label="Gross PFI share", color="#4472C4")
    ax.plot(years, annual_summary["net_measured_share_gdp_pct"], marker="o", label="Net measured share", color="#ED7D31")
    anchor_years = [year for year in years if year in GS_AI_CAPEX_SHARE_ANCHORS]
    ax.scatter(anchor_years, [GS_AI_CAPEX_SHARE_ANCHORS[year] for year in anchor_years], marker="x", s=80, color="#7030A0", label="Goldman diagnostic")
    shade_estimates(ax)
    ax.set_ylabel("Percent of nominal GDP")
    ax.set_title("AI investment level as a share of US GDP")
    ax.legend(frameon=False)
    fig.tight_layout(); fig.savefig(chart_paths[3], dpi=170); plt.close(fig)
else:
    save_pillow_line_chart(
        "Real GDP growth contribution",
        annual_summary[["real_gross_contribution_pp", "real_import_drag_pp", "real_net_contribution_pp"]],
        chart_paths[0], "pp",
        {"real_gross_contribution_pp": "Gross investment", "real_import_drag_pp": "Import drag", "real_net_contribution_pp": "Net measured"},
    )
    save_pillow_line_chart(
        "Company CAPEX, baseline, and AI excess",
        annual_summary[["reported_and_imputed_capex_bn", "flat_baseline_bn", "gross_ai_capex_bn"]],
        chart_paths[1], "$bn",
        {"reported_and_imputed_capex_bn": "Company CAPEX", "flat_baseline_bn": "Flat-2022 baseline", "gross_ai_capex_bn": "Gross AI excess"},
    )
    fallback_breakdown = industry_breakdown.loc[industry_breakdown["year"].between(2023, 2028)].pivot(index="year", columns="GICS_Industry_Group", values="us_ai_capex_source_bn").fillna(0)
    save_pillow_line_chart(
        "US AI CAPEX by industry", fallback_breakdown, chart_paths[2], "$bn",
        {
            "Automobiles & Components": "Tesla / auto",
            "Consumer Discretionary Distribution & Retail": "Amazon / retail",
            "Media & Entertainment": "Alphabet & Meta",
            "Software & Services": "Software & services",
            "Technology Hardware & Equipment": "Tech hardware",
            "Utilities": "Utilities",
        },
    )
    save_pillow_line_chart(
        "AI investment share of GDP",
        annual_summary[["gross_investment_share_gdp_pct", "net_measured_share_gdp_pct"]],
        chart_paths[3], "% GDP",
        {"gross_investment_share_gdp_pct": "Gross PFI share", "net_measured_share_gdp_pct": "Net measured share"},
    )
    ensure_fallback_year_labels(chart_paths[3], annual_summary.index)

print("Saved charts:")
for path in chart_paths:
    print(f"  {path.relative_to(ROOT)} ({path.stat().st_size:,} bytes)")

In [ ]:
# 14. Export a fully auditable Excel results package
assumptions_rows = [
    ("LAST_ACTUAL_QUARTER", str(LAST_ACTUAL_QUARTER), "Quarter"),
    ("MODEL_START", str(MODEL_START), "Quarter"),
    ("ANALYSIS_END", str(ANALYSIS_END), "Quarter"),
    ("TAIL_END", str(TAIL_END), "Quarter"),
    ("US_SHARE_DEFAULT", US_SHARE_DEFAULT, "%"),
    ("US_SHARE_UTILITY", US_SHARE_UTILITY, "%"),
    ("NOMINAL_GDP_GROWTH_ANNUAL", NOMINAL_GDP_GROWTH_ANNUAL, "%"),
    ("GDP_INFLATION_ANNUAL", gdp_inflation_annual, "%"),
    ("IMPUTATION_WARNING_SHARE", IMPUTATION_WARNING_SHARE, "%"),
    ("BASELINE_METHOD", "flat_2022", "Central scenario"),
    ("TREND_GROWTH_BOUNDS", str(TREND_GROWTH_ANNUAL_BOUNDS), "Annual rate"),
    ("COMPONENT_PRICE_BOUNDS", str(COMPONENT_PRICE_ANNUAL_BOUNDS), "Annual rate"),
    ("REAL_GDP_METHOD", "Additive chain-dollar approximation", "Method"),
]
for profile, row in CATEGORY_SPLITS.iterrows():
    for category, value in row.items():
        assumptions_rows.append((f"CATEGORY_SPLIT.{profile}.{category}", value, "%"))
for category, weights in LAG_WEIGHTS.items():
    assumptions_rows.append((f"LAG_WEIGHTS.{category}", json.dumps(weights), "Weights from t onward"))
for import_case in IMPORT_SHARES.columns:
    for category, value in IMPORT_SHARES[import_case].items():
        assumptions_rows.append((f"IMPORT_SHARE.{import_case}.{category}", value, "%"))
for company, profile in PROFILE_OVERRIDES.items():
    assumptions_rows.append((f"PROFILE_OVERRIDE.{company}", profile, "Capital profile"))
assumptions_output = pd.DataFrame(assumptions_rows, columns=["Assumption", "Value", "Units_or_note"])

company_output_columns = [
    "quarter", "Company_Name", "GICS_Industry_Group", "capital_profile", "data_status",
    "capex_reported", "capex_used", "baseline_flat_bn", "raw_excess_bn", "gross_ai_capex_bn",
    "unfloored_net_wedge_bn", "baseline_trend_bn", "trend_gross_ai_capex_bn", "us_share",
]
company_output = panel_output[company_output_columns].copy()
company_output["quarter"] = company_output["quarter"].astype(str)

quarterly_output = central_quarterly.loc[MODEL_START:ANALYSIS_END].reset_index().rename(columns={"index": "quarter"})
quarterly_output["quarter"] = quarterly_output["quarter"].astype(str)
asset_output = asset_flow_scenarios["flat_base"].copy()
asset_output["quarter"] = asset_output["quarter"].astype(str)
coverage_output = coverage_audit.loc[MODEL_START:ANALYSIS_END].reset_index()
coverage_output["quarter"] = coverage_output["quarter"].astype(str)

summary_columns = [
    "data_status", "reported_and_imputed_capex_bn", "flat_baseline_bn", "gross_ai_capex_bn",
    "us_ai_capex_source_bn", "gross_investment_nominal_saar_bn", "imports_nominal_saar_bn",
    "net_nominal_saar_bn", "nominal_net_contribution_pp", "real_gross_contribution_pp",
    "real_import_drag_pp", "real_net_contribution_pp", "gross_investment_share_gdp_pct",
    "net_measured_share_gdp_pct", "imputed_capex_share", "gs_us_ai_capex_anchor_bn", "gs_share_anchor_pct",
]
export_tables = {
    "Summary Annual": annual_summary[summary_columns].reset_index(),
    "Annual Detail": annual_summary.reset_index(),
    "Quarterly Contribution": quarterly_output,
    "Company AI CAPEX": company_output,
    "Industry Breakdown": industry_breakdown,
    "Asset Flows": asset_output,
    "Baseline Sensitivity": baseline_sensitivity,
    "Import Sensitivity": import_sensitivity,
    "Coverage Audit": coverage_output,
    "Baseline Audit": baseline_audit,
    "Cross Checks": cross_checks,
    "Assumptions": assumptions_output,
    "Sources Audit": sources_audit,
    "Checks": checks,
}

workbook_path = OUTPUT_DIR / "results.xlsx"
if EXPORT_OUTPUTS:
    from openpyxl import load_workbook
    from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
    from openpyxl.utils import get_column_letter

    with pd.ExcelWriter(workbook_path, engine="openpyxl") as writer:
        for sheet_name, frame in export_tables.items():
            frame.to_excel(writer, sheet_name=sheet_name[:31], index=False, startrow=3)

    workbook = load_workbook(workbook_path)
    dark_blue = "17365D"
    light_blue = "D9E2F3"
    pale_green = "E2F0D9"
    pale_red = "FCE4D6"
    thin_gray = Side(style="thin", color="D9E1F2")
    external_red = "FF0000"
    input_blue = "0000FF"

    for sheet_name, frame in export_tables.items():
        ws = workbook[sheet_name[:31]]
        ws.sheet_view.showGridLines = False
        ws["A1"] = sheet_name
        ws["A1"].font = Font(name="Aptos Display", size=16, bold=True, color=dark_blue)
        ws["A2"] = f"Data vintage: {OFFICIAL_DATA_VINTAGE} | Company cutoff: {LAST_ACTUAL_QUARTER} | Generated: {RUN_UTC}"
        ws["A2"].font = Font(name="Aptos", size=9, color="666666", italic=True)
        ws["A3"] = "Units are $bn unless otherwise labeled; contributions are percentage points."
        ws["A3"].font = Font(name="Aptos", size=9, color="666666")
        header_row = 4
        for cell in ws[header_row]:
            cell.fill = PatternFill("solid", fgColor=dark_blue)
            cell.font = Font(name="Aptos", bold=True, color="FFFFFF")
            cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        ws.freeze_panes = "A5"
        ws.auto_filter.ref = f"A{header_row}:{get_column_letter(max(ws.max_column, 1))}{ws.max_row}"
        for column_index, column_cells in enumerate(ws.iter_cols(min_row=header_row, max_row=min(ws.max_row, header_row + 250)), start=1):
            values = [str(cell.value) if cell.value is not None else "" for cell in column_cells]
            width = min(max(max((len(value) for value in values), default=0) + 2, 11), 32)
            ws.column_dimensions[get_column_letter(column_index)].width = width
        for row in ws.iter_rows(min_row=header_row + 1):
            for cell in row:
                cell.font = Font(name="Aptos", size=10)
                cell.alignment = Alignment(vertical="top")
                if isinstance(cell.value, (int, float)) and not isinstance(cell.value, bool):
                    cell.number_format = '#,##0.000;[Red](#,##0.000);-'
        header_lookup = {cell.value: cell.column for cell in ws[header_row] if cell.value is not None}
        for header, column_index in header_lookup.items():
            header_text = str(header)
            if header_text == "year":
                number_format = '0'
            elif header_text.endswith("_companies"):
                number_format = '#,##0'
            elif header_text in {"imputed_capex_share", "us_share"}:
                number_format = '0.0%'
            elif header_text.endswith("_pct") or header_text.endswith("_pp"):
                number_format = '0.000;[Red](0.000);-'
            elif header_text.endswith("_bn"):
                number_format = '#,##0.000;[Red](#,##0.000);-'
            else:
                continue
            for row_number in range(header_row + 1, ws.max_row + 1):
                ws.cell(row=row_number, column=column_index).number_format = number_format

    assumption_ws = workbook["Assumptions"]
    for cell in assumption_ws["B"][4:]:
        cell.font = Font(name="Aptos", size=10, color=input_blue)
    for row_number in range(5, assumption_ws.max_row + 1):
        if assumption_ws.cell(row=row_number, column=3).value == "%" and isinstance(assumption_ws.cell(row=row_number, column=2).value, (int, float)):
            assumption_ws.cell(row=row_number, column=2).number_format = "0.0%"
    sources_ws = workbook["Sources Audit"]
    for row in sources_ws.iter_rows(min_row=5):
        for cell in row:
            if isinstance(cell.value, str) and cell.value.startswith("http"):
                cell.font = Font(name="Aptos", size=10, color=external_red, underline="single")
    checks_ws = workbook["Checks"]
    status_column = [cell.value for cell in checks_ws[4]].index("Status") + 1
    for row in range(5, checks_ws.max_row + 1):
        status_cell = checks_ws.cell(row=row, column=status_column)
        status_cell.fill = PatternFill("solid", fgColor=pale_green if status_cell.value == "OK" else pale_red)
        status_cell.font = Font(name="Aptos", bold=True, color="006100" if status_cell.value == "OK" else "9C0006")
    workbook.save(workbook_path)
    print(f"Saved workbook: {workbook_path.relative_to(ROOT)} ({workbook_path.stat().st_size:,} bytes)")

In [ ]:
# 15. Final file and model-status verification
output_checks = pd.DataFrame(
    {
        "Output": ["results.xlsx"] + [path.name for path in chart_paths],
        "Exists": [workbook_path.exists()] + [path.exists() for path in chart_paths],
        "Bytes": [workbook_path.stat().st_size if workbook_path.exists() else 0] + [path.stat().st_size if path.exists() else 0 for path in chart_paths],
    }
)
display(output_checks)
print(f"Model checks: {(checks['Status'] == 'OK').sum()} OK, {(checks['Status'] != 'OK').sum()} REVIEW")
if not output_checks["Exists"].all():
    raise RuntimeError("One or more requested output files were not created.")

## Interpretation boundaries

- This is the direct **measured investment/import channel**, not a productivity, labor-income, consumption, or fiscal-multiplier model.
- Cash CAPEX does not consistently include model development, R&D, own-account software, finance leases, or SPV-financed assets. No mechanical intellectual-property adjustment is added.
- Semiconductor inputs may be treated as intermediate inputs in measured GDP even when they are integral to final AI infrastructure. The broader “true output” issue is disclosed but not added to the estimate.
- US shares, asset splits, recognition lags, import shares, baseline rules, and missing-estimate fills are modeling assumptions—not company disclosures.
- The real contribution subtracts constructed real flows from chain-dollar GDP. Because chained dollars are not strictly additive, it is an approximation; the notebook’s BEA computer comparison is a diagnostic rather than a calibration target.
- External Goldman and Bank of Italy estimates differ in perimeter and frequency. They are comparison points only and should not be treated as required outcomes.